# ST-A² Multi-Resolution Sweep: Area Attention vs Baseline

**Spatiotemporal Area Attention for V-JEPA 2 — H100 Optimized**

This notebook runs a **multi-resolution ablation** comparing:
- **Baseline**: Standard RoPE attention (full self-attention)
- **ST-A²**: RoPE Area Attention (partitioned spatiotemporal attention)

Both use the **real V-JEPA 2 model** (ViT-L encoder + predictor) with synthetic
random video tensors. No dataset download required.

## Resolution Sweep

| Config | Resolution | Frames | Total Tokens | Visible (~25%) | Per-Area (~128 each) | Batch |
|--------|-----------|--------|-------------|----------------|---------------------|-------|
| A | 256px | 16 | 2,048 | ~512 | ~128 | 4 |
| B | 256px | 64 | 8,192 | ~2,048 | ~512 | 1 |
| C | 384px | 16 | 4,608 | ~1,152 | ~288 | 2 |
| D | 384px | 64 | 18,432 | ~4,608 | ~1,152 | 1 |

**Goal**: Find the crossover point where ST-A² becomes faster than baseline.

**Hardware**: Lambda Labs 1×H100 (80GB), BF16.

In [ ]:
# Cell 1: Setup & Install
import os
if not os.path.exists('vjepa2'):
    !git clone -b feat/st-a2-area-attention https://github.com/tarassh/vjepa2.git
else:
    !cd vjepa2 && git pull origin feat/st-a2-area-attention
os.chdir('vjepa2')
!pip install -q timm
print('Setup complete.')

In [ ]:
# Cell 2: Imports & GPU Detection
import sys
import copy
import time
import gc
import traceback

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

if os.getcwd().endswith('vjepa2'):
    sys.path.insert(0, os.getcwd())
elif os.path.exists('vjepa2'):
    sys.path.insert(0, os.path.join(os.getcwd(), 'vjepa2'))

from app.vjepa.utils import init_video_model
from src.masks.multiseq_multiblock3d import _MaskGenerator
from src.masks.utils import apply_masks
from src.utils.logging import AverageMeter

assert torch.cuda.is_available(), 'CUDA required'
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
gpu_mem_gb = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9

# Auto-detect dtype: BF16 for Ampere+ (A100, H100), FP16 for older (T4, V100)
if torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
    dtype_str = 'bfloat16'
else:
    DTYPE = torch.float16
    dtype_str = 'float16'

print(f'GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'Dtype: {dtype_str}')
print(f'Compute capability: {props.major}.{props.minor}')

In [ ]:
# Cell 3: Multi-Resolution Configuration

# Shared model config (ViT-L, same as V-JEPA 2 pretrain)
MODEL_CFG = dict(
    model_name='vit_large',
    patch_size=16,
    tubelet_size=2,
    pred_depth=12,
    pred_embed_dim=384,
    pred_num_heads=12,
    num_steps=100,       # 100 steps per config (saves time vs 150)
    warmup_steps=10,
    lr=5.25e-4,
    weight_decay=0.04,
    loss_exp=1.0,
    ema_momentum=0.999,
)

# Resolution sweep configs
# Each entry: (label, crop_size, num_frames, batch_size)
SWEEP_RESOLUTIONS = [
    ('256px-16f', 256, 16, 4),   # A: 2,048 tokens, ~512 visible
    ('384px-16f', 384, 16, 2),   # C: 4,608 tokens, ~1,152 visible
    ('256px-64f', 256, 64, 1),   # B: 8,192 tokens, ~2,048 visible
    ('384px-64f', 384, 64, 1),   # D: 18,432 tokens, ~4,608 visible
]

# Build full config dicts for each resolution x [baseline, st_a2]
ALL_CONFIGS = {}
for label, crop, frames, batch in SWEEP_RESOLUTIONS:
    H = W = crop // MODEL_CFG['patch_size']
    T = frames // MODEL_CFG['tubelet_size']
    total = H * W * T
    visible = total // 4
    
    base = {
        **MODEL_CFG,
        'crop_size': crop,
        'num_frames': frames,
        'batch_size': batch,
        'total_tokens': total,
        'visible_tokens': visible,
        'resolution_label': label,
    }
    
    ALL_CONFIGS[(label, 'baseline')] = {
        **base,
        'use_area_attention': False,
    }
    ALL_CONFIGS[(label, 'st_a2')] = {
        **base,
        'use_area_attention': True,
        'area_attention_layers': [0, 18],
        'area_spatial_splits': 2,
        'area_temporal_splits': 2,
        'area_residual_scale': 1.0,
    }

# V-JEPA 2 mask config
MASK_CFGS = [
    dict(spatial_scale=(0.15, 0.15), temporal_scale=(1.0, 1.0),
         aspect_ratio=(0.75, 1.5), num_blocks=8, max_temporal_keep=1.0),
    dict(spatial_scale=(0.7, 0.7), temporal_scale=(1.0, 1.0),
         aspect_ratio=(0.75, 1.5), num_blocks=2, max_temporal_keep=1.0),
]

print(f'GPU: {gpu_name} ({gpu_mem_gb:.1f} GB), dtype: {dtype_str}')
print(f'Sweep: {len(SWEEP_RESOLUTIONS)} resolutions × 2 configs = {len(ALL_CONFIGS)} runs')
print(f'Steps per run: {MODEL_CFG["num_steps"]}')
print()
print(f'{"Label":<12} {"Crop":>5} {"Frames":>6} {"Batch":>5} {"Total":>7} {"Visible":>7} {"Per-Area":>8}')
print('-' * 60)
for label, crop, frames, batch in SWEEP_RESOLUTIONS:
    H = crop // 16
    T = frames // 2
    total = H * H * T
    visible = total // 4
    per_area = visible // 4
    print(f'{label:<12} {crop:>5} {frames:>6} {batch:>5} {total:>7} {visible:>7} {per_area:>8}')

In [ ]:
# Cell 4: Synthetic Data & Mask Generator

def make_mask_generators(cfg):
    generators = []
    for m in MASK_CFGS:
        gen = _MaskGenerator(
            crop_size=cfg['crop_size'],
            num_frames=cfg['num_frames'],
            spatial_patch_size=cfg['patch_size'],
            temporal_patch_size=cfg['tubelet_size'],
            spatial_pred_mask_scale=m['spatial_scale'],
            temporal_pred_mask_scale=m['temporal_scale'],
            aspect_ratio=m['aspect_ratio'],
            npred=m['num_blocks'],
            max_context_frames_ratio=m['max_temporal_keep'],
        )
        generators.append(gen)
    return generators


def make_synthetic_batch(cfg, mask_generators):
    B = cfg['batch_size']
    T = cfg['num_frames']
    H = W = cfg['crop_size']
    clip = torch.randn(B, 3, T, H, W, device=device)
    all_masks_enc = []
    all_masks_pred = []
    for gen in mask_generators:
        masks_enc, masks_pred = gen(B)
        all_masks_enc.append(masks_enc.to(device))
        all_masks_pred.append(masks_pred.to(device))
    return [clip], [all_masks_enc], [all_masks_pred]

print('Data utilities defined.')

In [ ]:
# Cell 5: Model Builder

def build_models(cfg):
    num_mask_tokens = len(MASK_CFGS)
    encoder, predictor = init_video_model(
        device=device,
        patch_size=cfg['patch_size'],
        max_num_frames=cfg['num_frames'],
        tubelet_size=cfg['tubelet_size'],
        model_name=cfg['model_name'],
        crop_size=cfg['crop_size'],
        pred_depth=cfg['pred_depth'],
        pred_num_heads=cfg['pred_num_heads'],
        pred_embed_dim=cfg['pred_embed_dim'],
        uniform_power=True,
        use_mask_tokens=True,
        num_mask_tokens=num_mask_tokens,
        zero_init_mask_tokens=True,
        use_sdpa=True,
        use_rope=True,
        use_activation_checkpointing=True,
        use_area_attention=cfg['use_area_attention'],
        area_attention_layers=cfg.get('area_attention_layers'),
        area_spatial_splits=cfg.get('area_spatial_splits', 2),
        area_temporal_splits=cfg.get('area_temporal_splits', 2),
        area_residual_scale=cfg.get('area_residual_scale', 1.0),
    )
    target_encoder = copy.deepcopy(encoder)
    target_encoder.to(device)
    for p in target_encoder.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(predictor.parameters()),
        lr=cfg['lr'], weight_decay=cfg['weight_decay'], betas=(0.9, 0.999),
    )
    scaler = torch.amp.GradScaler('cuda')
    enc_params = sum(p.numel() for p in encoder.parameters()) / 1e6
    pred_params = sum(p.numel() for p in predictor.parameters()) / 1e6
    print(f'  Encoder: {enc_params:.1f}M params, Predictor: {pred_params:.1f}M params')
    return encoder, predictor, target_encoder, optimizer, scaler

print('build_models() defined.')

In [ ]:
# Cell 6: Training Step

def train_step(encoder, predictor, target_encoder, optimizer, scaler,
               clips, masks_enc, masks_pred, loss_exp=1.0, momentum=0.999):
    def forward_target(c):
        with torch.no_grad():
            h = target_encoder(c)
            h = [F.layer_norm(hi, (hi.size(-1),)) for hi in h]
            return h

    def forward_context(c):
        z = encoder(c, masks_enc)
        z = predictor(z, masks_enc, masks_pred)
        return z

    def loss_fn(z, h):
        h = [apply_masks(hi, mi, concat=False) for hi, mi in zip(h, masks_pred)]
        loss, n = 0, 0
        for zi, hi in zip(z, h):
            for zij, hij in zip(zi, hi):
                loss += torch.mean(torch.abs(zij - hij) ** loss_exp) / loss_exp
                n += 1
        loss /= n
        return loss

    with torch.amp.autocast('cuda', dtype=DTYPE):
        h = forward_target(clips)
        z = forward_context(clips)
        loss = loss_fn(z, h)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()

    with torch.no_grad():
        for param_q, param_k in zip(encoder.parameters(), target_encoder.parameters()):
            param_k.data.mul_(momentum).add_(param_q.data, alpha=1 - momentum)

    return float(loss)

print('train_step() defined.')

In [ ]:
# Cell 7: Run Ablation

def run_ablation(name, cfg):
    label = cfg['resolution_label']
    attn_type = 'ST-A\u00b2' if cfg['use_area_attention'] else 'Baseline'
    print(f'\n{"="*70}')
    print(f'Running: {label} / {attn_type}')
    print(f'  Resolution: {cfg["crop_size"]}px, {cfg["num_frames"]}f, batch={cfg["batch_size"]}')
    print(f'  Tokens: {cfg["total_tokens"]} total, ~{cfg["visible_tokens"]} visible')
    print(f'  Area attention: {cfg["use_area_attention"]}')
    print(f'{"="*70}')

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    encoder, predictor, target_encoder, optimizer, scaler = build_models(cfg)
    mask_generators = make_mask_generators(cfg)

    num_steps = cfg['num_steps']
    losses = []
    step_times_ms = []

    print('  Warmup (3 steps)...')
    for _ in range(3):
        clips, masks_enc, masks_pred = make_synthetic_batch(cfg, mask_generators)
        _ = train_step(encoder, predictor, target_encoder, optimizer, scaler,
                       clips, masks_enc, masks_pred,
                       loss_exp=cfg['loss_exp'], momentum=cfg['ema_momentum'])
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    print(f'  Training ({num_steps} steps)...')
    for step in range(num_steps):
        clips, masks_enc, masks_pred = make_synthetic_batch(cfg, mask_generators)
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        start_event.record()
        loss = train_step(encoder, predictor, target_encoder, optimizer, scaler,
                          clips, masks_enc, masks_pred,
                          loss_exp=cfg['loss_exp'], momentum=cfg['ema_momentum'])
        end_event.record()
        torch.cuda.synchronize()
        elapsed_ms = start_event.elapsed_time(end_event)
        losses.append(loss)
        step_times_ms.append(elapsed_ms)
        if (step + 1) % 25 == 0 or step == 0:
            print(f'    Step {step+1:4d}/{num_steps}: loss={np.mean(losses[-25:]):.4f}, time={np.mean(step_times_ms[-25:]):.1f}ms')

    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2
    del encoder, predictor, target_encoder, optimizer, scaler
    torch.cuda.empty_cache()
    gc.collect()

    result = {
        'losses': losses,
        'step_times_ms': step_times_ms,
        'peak_mem_mb': peak_mem_mb,
        'avg_step_ms': np.mean(step_times_ms),
        'final_loss': np.mean(losses[-20:]),
        'throughput_steps_sec': 1000.0 / np.mean(step_times_ms),
        'resolution_label': cfg['resolution_label'],
        'visible_tokens': cfg['visible_tokens'],
    }
    print(f'  Done. loss={result["final_loss"]:.4f}, '
          f'time={result["avg_step_ms"]:.1f}ms, '
          f'mem={result["peak_mem_mb"]:.0f}MB')
    return result

print('run_ablation() defined.')

In [ ]:
# Cell 8: Execute Full Sweep
#
# Runs all resolution x config combinations.
# OOM-safe: skips configs that don't fit in GPU memory.

results = {}
skipped = []

for res_label, crop, frames, batch in SWEEP_RESOLUTIONS:
    for config_name in ['baseline', 'st_a2']:
        key = (res_label, config_name)
        cfg = ALL_CONFIGS[key]
        try:
            results[key] = run_ablation(config_name, cfg)
        except torch.cuda.OutOfMemoryError:
            print(f'\n  \u274c OOM: {res_label}/{config_name} — skipping')
            skipped.append(key)
            torch.cuda.empty_cache()
            gc.collect()
        except Exception as e:
            print(f'\n  \u274c Error: {res_label}/{config_name} — {e}')
            traceback.print_exc()
            skipped.append(key)
            torch.cuda.empty_cache()
            gc.collect()

print(f'\n\n{"="*70}')
print(f'Sweep complete! {len(results)} runs succeeded, {len(skipped)} skipped.')
if skipped:
    print(f'Skipped: {skipped}')

In [ ]:
# Cell 9: Per-Layer Profiling

from src.models.utils.modules import Block, RoPEAttention, RoPEAreaAttention

def profile_encoder(cfg, num_runs=20):
    attn_type = 'ST-A\u00b2' if cfg['use_area_attention'] else 'Baseline'
    label = cfg['resolution_label']
    print(f'\nProfiling: {label} / {attn_type}')

    torch.cuda.empty_cache()
    gc.collect()

    encoder, predictor = init_video_model(
        device=device,
        patch_size=cfg['patch_size'],
        max_num_frames=cfg['num_frames'],
        tubelet_size=cfg['tubelet_size'],
        model_name=cfg['model_name'],
        crop_size=cfg['crop_size'],
        pred_depth=cfg['pred_depth'],
        pred_num_heads=cfg['pred_num_heads'],
        pred_embed_dim=cfg['pred_embed_dim'],
        uniform_power=True,
        use_mask_tokens=True,
        num_mask_tokens=len(MASK_CFGS),
        zero_init_mask_tokens=True,
        use_sdpa=True,
        use_rope=True,
        use_activation_checkpointing=False,
        use_area_attention=cfg['use_area_attention'],
        area_attention_layers=cfg.get('area_attention_layers'),
        area_spatial_splits=cfg.get('area_spatial_splits', 2),
        area_temporal_splits=cfg.get('area_temporal_splits', 2),
        area_residual_scale=cfg.get('area_residual_scale', 1.0),
    )
    encoder.eval()

    blocks = encoder.backbone.blocks
    num_layers = len(blocks)
    layer_timings = [{'attn': [], 'mlp': []} for _ in range(num_layers)]

    original_forwards = []
    for i, block in enumerate(blocks):
        original_forwards.append(block.forward)
        def make_profiled_forward(block_ref, layer_idx):
            def profiled_forward(x, mask=None, attn_mask=None, T=None, H_patches=None, W_patches=None):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                if isinstance(block_ref.attn, (RoPEAttention, RoPEAreaAttention)):
                    y = block_ref.attn(block_ref.norm1(x), mask=mask, attn_mask=attn_mask,
                                       T=T, H_patches=H_patches, W_patches=W_patches)
                else:
                    y = block_ref.attn(block_ref.norm1(x), mask=mask, attn_mask=attn_mask)
                torch.cuda.synchronize()
                t1 = time.perf_counter()
                x_out = x + block_ref.drop_path(y)
                torch.cuda.synchronize()
                t2 = time.perf_counter()
                x_out = x_out + block_ref.drop_path(block_ref.mlp(block_ref.norm2(x_out)))
                torch.cuda.synchronize()
                t3 = time.perf_counter()
                layer_timings[layer_idx]['attn'].append((t1 - t0) * 1000)
                layer_timings[layer_idx]['mlp'].append((t3 - t2) * 1000)
                return x_out
            return profiled_forward
        block.forward = make_profiled_forward(block, i)

    mask_generators = make_mask_generators(cfg)

    print('  Warmup (3 runs)...')
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
        for _ in range(3):
            clips, masks_enc, _ = make_synthetic_batch(cfg, mask_generators)
            _ = encoder(clips, masks_enc)
    for lt in layer_timings:
        lt['attn'].clear()
        lt['mlp'].clear()

    print(f'  Profiling ({num_runs} forward passes)...')
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
        for _ in range(num_runs):
            clips, masks_enc, _ = make_synthetic_batch(cfg, mask_generators)
            _ = encoder(clips, masks_enc)

    for i, block in enumerate(blocks):
        block.forward = original_forwards[i]

    profile_data = []
    total_attn_ms = 0
    total_mlp_ms = 0
    for i in range(num_layers):
        attn_ms = np.mean(layer_timings[i]['attn'])
        mlp_ms = np.mean(layer_timings[i]['mlp'])
        attn_type_name = type(blocks[i].attn).__name__
        total_attn_ms += attn_ms
        total_mlp_ms += mlp_ms
        profile_data.append({
            'layer': i, 'attn_type': attn_type_name,
            'attn_ms': attn_ms, 'mlp_ms': mlp_ms,
            'total_ms': attn_ms + mlp_ms,
            'attn_pct': attn_ms / (attn_ms + mlp_ms) * 100,
        })

    del encoder, predictor
    torch.cuda.empty_cache()
    gc.collect()

    total_ms = total_attn_ms + total_mlp_ms
    print(f'  Encoder: {total_ms:.1f}ms (attn={total_attn_ms:.1f}ms [{total_attn_ms/total_ms*100:.0f}%], mlp={total_mlp_ms:.1f}ms [{total_mlp_ms/total_ms*100:.0f}%])')
    return profile_data

# Run profiling for all completed configs
profile_results = {}
for key in results:
    cfg = ALL_CONFIGS[key]
    try:
        profile_results[key] = profile_encoder(cfg)
    except torch.cuda.OutOfMemoryError:
        print(f'  \u274c OOM during profiling: {key} — skipping')
        torch.cuda.empty_cache()
        gc.collect()

print('\nAll profiling complete!')

In [ ]:
# Cell 10: Summary Table — All Resolutions

print('\n' + '=' * 100)
print('  ST-A\u00b2 MULTI-RESOLUTION SWEEP RESULTS')
print('=' * 100)
print(f'  GPU: {gpu_name} ({gpu_mem_gb:.1f} GB), dtype: {dtype_str}')
print(f'  Model: {MODEL_CFG["model_name"]}, steps: {MODEL_CFG["num_steps"]}')
print()

header = (f'{"Resolution":<12} {"Visible":>7} {"Batch":>5} '
          f'{"BL Time":>9} {"ST Time":>9} {"\u0394 Time":>9} '
          f'{"BL Loss":>9} {"ST Loss":>9} {"\u0394 Loss":>9} '
          f'{"BL Mem":>8} {"ST Mem":>8}')
print(header)
print('-' * 100)

for res_label, crop, frames, batch in SWEEP_RESOLUTIONS:
    bl_key = (res_label, 'baseline')
    st_key = (res_label, 'st_a2')
    
    if bl_key not in results or st_key not in results:
        H = crop // 16
        T = frames // 2
        vis = H * H * T // 4
        status = 'SKIPPED (OOM)' if bl_key in skipped or st_key in skipped else 'SKIPPED'
        print(f'{res_label:<12} {vis:>7} {batch:>5}   {status}')
        continue
    
    bl = results[bl_key]
    st = results[st_key]
    vis = bl['visible_tokens']
    
    dt = (st['avg_step_ms'] - bl['avg_step_ms']) / bl['avg_step_ms'] * 100
    dl = (st['final_loss'] - bl['final_loss']) / bl['final_loss'] * 100
    
    # Mark crossover with arrow
    speed_marker = '\u2705' if dt <= 0 else ''
    
    print(f'{res_label:<12} {vis:>7} {batch:>5} '
          f'{bl["avg_step_ms"]:>8.1f}ms {st["avg_step_ms"]:>8.1f}ms {dt:>+8.1f}% '
          f'{bl["final_loss"]:>9.4f} {st["final_loss"]:>9.4f} {dl:>+8.1f}% '
          f'{bl["peak_mem_mb"]:>7.0f}M {st["peak_mem_mb"]:>7.0f}M '
          f'{speed_marker}')

print('=' * 100)
print()
print('\u2705 = ST-A\u00b2 is FASTER than baseline')
print('Negative \u0394 Time = ST-A\u00b2 faster, Negative \u0394 Loss = ST-A\u00b2 converges better')

In [ ]:
# Cell 11: Multi-Resolution Charts

# Collect data for plotting
plot_data = []
for res_label, crop, frames, batch in SWEEP_RESOLUTIONS:
    bl_key = (res_label, 'baseline')
    st_key = (res_label, 'st_a2')
    if bl_key in results and st_key in results:
        bl = results[bl_key]
        st = results[st_key]
        plot_data.append({
            'label': res_label,
            'visible': bl['visible_tokens'],
            'bl_time': bl['avg_step_ms'],
            'st_time': st['avg_step_ms'],
            'bl_loss': bl['final_loss'],
            'st_loss': st['final_loss'],
            'bl_mem': bl['peak_mem_mb'],
            'st_mem': st['peak_mem_mb'],
            'speedup': bl['avg_step_ms'] / st['avg_step_ms'],
        })

if not plot_data:
    print('No data to plot!')
else:
    vis = [d['visible'] for d in plot_data]
    labels = [d['label'] for d in plot_data]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Top-left: Step time vs visible tokens
    ax = axes[0, 0]
    ax.plot(vis, [d['bl_time'] for d in plot_data], 'o-', color='#2196F3',
            linewidth=2, markersize=8, label='Baseline')
    ax.plot(vis, [d['st_time'] for d in plot_data], 's-', color='#FF5722',
            linewidth=2, markersize=8, label='ST-A\u00b2')
    for i, d in enumerate(plot_data):
        ax.annotate(d['label'], (vis[i], d['bl_time']), fontsize=8,
                    textcoords='offset points', xytext=(5, 5))
    ax.set_xlabel('Visible Tokens')
    ax.set_ylabel('Avg Step Time (ms)')
    ax.set_title('Step Time vs Token Count')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Top-right: Loss vs visible tokens
    ax = axes[0, 1]
    ax.plot(vis, [d['bl_loss'] for d in plot_data], 'o-', color='#2196F3',
            linewidth=2, markersize=8, label='Baseline')
    ax.plot(vis, [d['st_loss'] for d in plot_data], 's-', color='#FF5722',
            linewidth=2, markersize=8, label='ST-A\u00b2')
    for i, d in enumerate(plot_data):
        ax.annotate(d['label'], (vis[i], d['bl_loss']), fontsize=8,
                    textcoords='offset points', xytext=(5, 5))
    ax.set_xlabel('Visible Tokens')
    ax.set_ylabel('Final Loss (L1)')
    ax.set_title('Loss vs Token Count')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Bottom-left: Memory comparison
    ax = axes[1, 0]
    x_pos = np.arange(len(plot_data))
    w = 0.35
    ax.bar(x_pos - w/2, [d['bl_mem'] for d in plot_data], w, color='#2196F3',
           label='Baseline', alpha=0.85)
    ax.bar(x_pos + w/2, [d['st_mem'] for d in plot_data], w, color='#FF5722',
           label='ST-A\u00b2', alpha=0.85)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Peak Memory (MB)')
    ax.set_title('Peak GPU Memory')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    # Bottom-right: Speedup ratio
    ax = axes[1, 1]
    speedups = [d['speedup'] for d in plot_data]
    colors = ['#4CAF50' if s >= 1.0 else '#F44336' for s in speedups]
    ax.bar(labels, speedups, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1.5, label='Break-even')
    for i, s in enumerate(speedups):
        ax.text(i, s + 0.01, f'{s:.2f}x', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('Speedup (Baseline / ST-A\u00b2)')
    ax.set_title('ST-A\u00b2 Speedup Ratio')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(bottom=min(0.8, min(speedups) - 0.05))

    plt.suptitle(f'ST-A\u00b2 Multi-Resolution Sweep — {gpu_name} ({dtype_str})',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('sweep_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: sweep_results.png')

In [ ]:
# Cell 12: Per-Layer Profiling Table (highest resolution)

# Find highest resolution that has profiling data
best_res = None
for res_label, _, _, _ in reversed(SWEEP_RESOLUTIONS):
    if (res_label, 'baseline') in profile_results and (res_label, 'st_a2') in profile_results:
        best_res = res_label
        break

if best_res is None:
    print('No profiling data available.')
else:
    print(f'\nDetailed profiling for: {best_res}')
    print('=' * 90)

    for config_name, label in [('baseline', 'BASELINE'), ('st_a2', 'ST-A\u00b2')]:
        key = (best_res, config_name)
        data = profile_results[key]
        total_attn = sum(d['attn_ms'] for d in data)
        total_mlp = sum(d['mlp_ms'] for d in data)
        total = total_attn + total_mlp

        print(f'\n  {label}')
        print(f'  {"Layer":<6} {"Type":<22} {"Attn(ms)":>9} {"MLP(ms)":>9} {"Total(ms)":>10} {"Attn%":>7}')
        print(f'  {"-"*68}')
        for d in data:
            print(f'  {d["layer"]:<6} {d["attn_type"]:<22} {d["attn_ms"]:>9.2f} {d["mlp_ms"]:>9.2f} '
                  f'{d["total_ms"]:>10.2f} {d["attn_pct"]:>6.1f}%')
        print(f'  {"-"*68}')
        print(f'  {"TOTAL":<6} {"":<22} {total_attn:>9.2f} {total_mlp:>9.2f} '
              f'{total:>10.2f} {total_attn/total*100:>6.1f}%')

    # Summary comparison
    bl_data = profile_results[(best_res, 'baseline')]
    st_data = profile_results[(best_res, 'st_a2')]
    bl_attn = sum(d['attn_ms'] for d in bl_data)
    st_attn = sum(d['attn_ms'] for d in st_data)
    bl_total = sum(d['total_ms'] for d in bl_data)
    st_total = sum(d['total_ms'] for d in st_data)
    
    print(f'\n  Encoder total: Baseline={bl_total:.1f}ms, ST-A\u00b2={st_total:.1f}ms '
          f'({(st_total-bl_total)/bl_total*100:+.1f}%)')
    print(f'  Attention:     Baseline={bl_attn:.1f}ms, ST-A\u00b2={st_attn:.1f}ms '
          f'({(st_attn-bl_attn)/bl_attn*100:+.1f}%)')

In [ ]:
# Cell 13: CSV Export — All Results

# Per-step metrics
rows = []
for key, r in results.items():
    res_label, config_name = key
    cfg = ALL_CONFIGS[key]
    for i in range(len(r['losses'])):
        rows.append({
            'resolution': res_label,
            'config': config_name,
            'crop_size': cfg['crop_size'],
            'num_frames': cfg['num_frames'],
            'batch_size': cfg['batch_size'],
            'visible_tokens': cfg['visible_tokens'],
            'step': i + 1,
            'loss': r['losses'][i],
            'step_time_ms': r['step_times_ms'][i],
        })
df_steps = pd.DataFrame(rows)
df_steps.to_csv('sweep_results.csv', index=False)
print(f'Per-step metrics: sweep_results.csv ({len(df_steps)} rows)')

# Summary
summary_rows = []
for key, r in results.items():
    res_label, config_name = key
    cfg = ALL_CONFIGS[key]
    summary_rows.append({
        'resolution': res_label,
        'config': config_name,
        'model': cfg['model_name'],
        'crop_size': cfg['crop_size'],
        'num_frames': cfg['num_frames'],
        'batch_size': cfg['batch_size'],
        'total_tokens': cfg['total_tokens'],
        'visible_tokens': cfg['visible_tokens'],
        'use_area_attention': cfg['use_area_attention'],
        'num_steps': cfg['num_steps'],
        'dtype': dtype_str,
        'gpu': gpu_name,
        'final_loss': r['final_loss'],
        'avg_step_ms': r['avg_step_ms'],
        'peak_mem_mb': r['peak_mem_mb'],
        'throughput_steps_sec': r['throughput_steps_sec'],
    })
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv('sweep_summary.csv', index=False)
print(f'Summary: sweep_summary.csv')
print()
print(df_summary.to_string(index=False))
print()
print('Done! Download sweep_results.csv and sweep_summary.csv for analysis.')